
# 🧪 Teacher RAG — Minimal Notebook (Gemini + LangChain + Chroma)

This notebook gives you a **from-scratch** Retrieval-Augmented Generation (RAG) prototype for a *Teacher Assistant* use case.

**You will be able to:**
- Ingest materials (CSV text or files like PDF/TXT) into a Chroma vector DB
- Ask grounded questions with **citations**
- Generate a **case study** grounded in your materials

> **Requirements:** a Google API key with access to Gemini. Set `GOOGLE_API_KEY` before running.


## 1) Install packages

In [18]:
!pip install -q playwright
!python -m playwright install chromium


[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
129.7 MiB [                    ] 0% 0.0s129.7 MiB [                    ] 0% 1003.3s129.7 MiB [                    ] 0% 649.2s129.7 MiB [                    ] 0% 400.3s129.7 MiB [                    ] 0% 241.7s129.7 MiB [                    ] 0% 160.6s129.7 MiB [                    ] 0% 94.9s129.7 MiB [                    ] 0% 54.0s129.7 MiB [                    ] 0% 56.4s129.7 MiB [                    ] 1% 41.0s129.7 MiB [                    ] 1% 32.8s129.7 MiB [                    ] 1% 41.4s129.7 MiB [                    ] 1% 44.0s129.7 MiB [                    ] 1% 45.4s129.7 MiB [                    ] 2% 34.2s129.7 MiB [                    ] 2% 36.8s129.7 MiB [                    ] 2% 39.7s129.7 MiB [                    ] 2% 43.2s129.7 MiB [                    ] 2% 48.5s129.7 MiB [                    ] 2% 53.8s129.7 MiB [                    ] 2% 59.8s129.7 MiB [            

In [19]:
from langchain import PromptTemplate
from langchain import hub
from langchain.docstore.document import Document
from langchain.document_loaders import WebBaseLoader
from langchain.schema import StrOutputParser
from langchain.schema.prompt_template import format_document
from langchain.schema.runnable import RunnablePassthrough
from langchain.vectorstores import Chroma
from langchain.document_loaders import DataFrameLoader
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os, time, re, pandas as pd, pathlib, datetime as dt
from urllib.parse import urljoin, urlparse
import requests
from bs4 import BeautifulSoup
from playwright.sync_api import sync_playwright

## 2) Environment & paths

In [60]:
# Run this cell and paste the API key in the prompt
import os
import getpass

os.environ['GOOGLE_API_KEY'] = getpass.getpass('Gemini API Key:')

# Retrival Data

In [14]:
TARGET_URL = "https://www.dayofaiusa.org/curriculum/grades-3-5"
ROOT       = "https://www.dayofaiusa.org/"
CSV_PATH   = "dayofaiusa_grades3_5.csv"
DB_DIR     = "./chroma_db_dayofai_org"

session = requests.Session()
session.headers.update({"User-Agent":"Mozilla/5.0 (Mac) TeacherRAG/1.0"})


In [15]:
html = ""
try:
    r = session.get(TARGET_URL, timeout=20)
    if r.status_code == 200 and "text/html" in r.headers.get("Content-Type",""):
        html = r.text
except Exception as e:
    print("plain fetch failed:", e)

soup = BeautifulSoup(html, "html.parser") if html else None
plain_len = len(soup.get_text(" ", strip=True)) if soup else 0
print("Plain-text length:", plain_len)
print((soup.get_text(" ", strip=True)[:400] if soup else "") or "[no plain text]")


Plain-text length: 276
Day of AI USA Grades 3-5 (ages 8-10) Lessons Loading lesson titles... Educator Guide Educator Guide Slides 🔒 Student Resources 🔒 Tutorial Videos 🔒 Copy Please register to gain access to all of the resources. Loading lesson content... © 2025 Day of AI USA. All rights reserved.


In [21]:
from playwright.async_api import async_playwright

if plain_len < 400:
    async def render(url: str) -> str:
        async with async_playwright() as p:
            browser  = await p.chromium.launch(headless=True)
            context  = await browser.new_context()
            page     = await context.new_page()
            await page.goto(url, timeout=45_000, wait_until="networkidle")
            await page.wait_for_timeout(1200)  # small extra wait
            html     = await page.content()
            await browser.close()
            return html

    # In Jupyter you can await directly:
    html = await render(TARGET_URL)
    soup = BeautifulSoup(html, "html.parser")
    print("Rendered text length:", len(soup.get_text(" ", strip=True)))


Rendered text length: 10486


In [12]:
from urllib.parse import urljoin

CURRIC = "https://www.dayofaiusa.org/curriculum"
html2 = s.get(CURRIC, timeout=20).text
soup2 = BeautifulSoup(html2, "html.parser")

sections = []
current = None
for el in soup2.find_all(["h2","p","a"]):
    t = el.get_text(" ", strip=True)
    if not t:
        continue
    if el.name == "h2" and "Grades" in t:
        current = {"Band": t, "Summary": "", "CTA": ""}
        sections.append(current)
    elif el.name == "p" and current and not current["Summary"]:
        current["Summary"] = t
    elif el.name == "a" and current and t.lower().startswith("view lessons"):
        current["CTA"] = urljoin(CURRIC, el.get("href",""))

df_public = pd.DataFrame(sections)
df_public


,Band,Summary,CTA
0,Grades K-2 (ages 5-7),Our early childhood curriculum introduces youn...,https://www.dayofaiusa.org/curriculum/grades-k-2
1,Grades 3-5 (ages 8-10),Our upper elementary curriculum builds on thes...,https://www.dayofaiusa.org/curriculum/grades-3-5
2,Grades 6-8 (ages 11-13),Our middle school curriculum expands students'...,https://www.dayofaiusa.org/curriculum/grades-6-8
3,Grades 9-12 (ages 14+),Our high school curriculum prepares students f...,https://www.dayofaiusa.org/curriculum/grades-9-12


In [ ]:
out_csv = "dayofaiusa_curriculum_summaries.csv"
df_public.to_csv(out_csv, index=False)
print("Saved:", out_csv)

In [22]:
# strip non-content
for tag in soup(["script","style","noscript","iframe","svg"]):
    tag.decompose()

main = soup.find(["main","article"]) or soup.select_one("[role=main]") or soup.body or soup
for tag in main.find_all(["nav","header","footer","aside"]):
    tag.decompose()

parts = []
for elm in main.find_all(["h1","h2","h3","p","li","blockquote","pre"]):
    t = elm.get_text(" ", strip=True)
    if t:
        parts.append(t)

page_text = "\n".join(parts).strip()
page_text = re.sub(r"\n{3,}", "\n\n", page_text)
page_text = re.sub(r"[ \t]{2,}", " ", page_text)

page_title   = soup.title.get_text(" ", strip=True) if soup.title else "Untitled"
h1 = main.find("h1"); h2 = main.find("h2")
page_section = (h1.get_text(" ", strip=True) if h1 else "") or (h2.get_text(" ", strip=True) if h2 else "")

rows = []
if len(page_text) > 200:
    rows.append({
        "Title":   page_title or page_section or "Untitled",
        "URL":     TARGET_URL,
        "Section": page_section,
        "Text":    page_text
    })

len(rows), (rows[0]["Title"] if rows else None)


(1, 'Day of AI USA')

In [24]:
rows

[{'Title': 'Day of AI USA',
  'URL': 'https://www.dayofaiusa.org/curriculum/grades-3-5',
  'Section': 'Lesson 1: What is AI? | 45-60 minutes',
  'Text': 'Please register to gain access to all of the resources.\nLesson 1: What is AI? | 45-60 minutes\nWhat do we mean by Artificial Intelligence?\nObjectives\nIn this lesson, we will learn what it means to have artificial intelligence.\nWe will learn to tell if a device or item uses artificial intelligence.\nVocabulary\nArtificial intelligence, n. a program made by people that makes computers do things that seem intelligent (or smart) in the same way that humans are intelligent\nArtificial, adj. made by humans, especially in imitation of something natural\nIntelligence, n. the ability to learn or understand\nUnderstand, v. to grasp the meaning of\nPerceive, v. to become aware of, know, or identify through one of the senses (sight, taste, smell, hear, touch)\nInteract, v. to communicate with or react\nGenerative, adj. relating to or characte

In [25]:
df = pd.DataFrame(rows, columns=["Title","URL","Section","Text"])
print("Rows:", len(df))
df.head(1)

Rows: 1


,Title,URL,Section,Text
0,Day of AI USA,https://www.dayofaiusa.org/curriculum/grades-3-5,Lesson 1: What is AI? | 45-60 minutes,Please register to gain access to all of the r...


In [ ]:
out_csv = f"dayofaiusa_grades3_5_{dt.date.today().isoformat()}.csv"
df.to_csv(out_csv, index=False)
print("Saved CSV to:", out_csv)


In [26]:
out_csv = f"dayofaiusa_grades3_5_{dt.date.today().isoformat()}.csv"
df.to_csv(out_csv, index=False)
print("Saved CSV to:", out_csv)


Saved CSV to: dayofaiusa_grades3_5_2025-10-15.csv


In [27]:
from langchain_community.document_loaders import DataFrameLoader

loader = DataFrameLoader(df, page_content_column="Text")
docs = loader.load()
print("Docs:", len(docs))


Docs: 1


In [35]:
docs[0]

Document(metadata={'Title': 'Day of AI USA', 'URL': 'https://www.dayofaiusa.org/curriculum/grades-3-5', 'Section': 'Lesson 1: What is AI? | 45-60 minutes'}, page_content='Please register to gain access to all of the resources.\nLesson 1: What is AI? | 45-60 minutes\nWhat do we mean by Artificial Intelligence?\nObjectives\nIn this lesson, we will learn what it means to have artificial intelligence.\nWe will learn to tell if a device or item uses artificial intelligence.\nVocabulary\nArtificial intelligence, n. a program made by people that makes computers do things that seem intelligent (or smart) in the same way that humans are intelligent\nArtificial, adj. made by humans, especially in imitation of something natural\nIntelligence, n. the ability to learn or understand\nUnderstand, v. to grasp the meaning of\nPerceive, v. to become aware of, know, or identify through one of the senses (sight, taste, smell, hear, touch)\nInteract, v. to communicate with or react\nGenerative, adj. relati

In [36]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain.vectorstores import Chroma
from langchain.prompts import PromptTemplate


In [40]:
import os
assert os.getenv("GOOGLE_API_KEY"), "GOOGLE_API_KEY is not set in this kernel."
print("API key found ✅")


API key found ✅


In [41]:
assert len(docs) > 0, "Docs list is empty—check your df and page_content_column."
print("Docs loaded:", len(docs))

Docs loaded: 1


In [42]:
from playwright.async_api import async_playwright

if plain_len < 400:
    async def render(url: str) -> str:
        async with async_playwright() as p:
            browser  = await p.chromium.launch(headless=True)
            context  = await browser.new_context()
            page     = await context.new_page()
            await page.goto(url, timeout=45_000, wait_until="networkidle")
            await page.wait_for_timeout(1200)  # small extra wait
            html     = await page.content()
            await browser.close()
            return html

    # In Jupyter you can await directly:
    html = await render(TARGET_URL)
    soup = BeautifulSoup(html, "html.parser")
    print("Rendered text length:", len(soup.get_text(" ", strip=True)))


Rendered text length: 10486


In [43]:
vectorstore_disk = Chroma(
    persist_directory=DB_DIR,
    embedding_function=gemini_embeddings  # <-- use 'embedding_function=' when reopening
)
retriever = vectorstore_disk.as_retriever(search_kwargs={"k": 5})
print("Retriever ready ✅")


Retriever ready ✅


/var/folders/xr/69vk_5_966g8wxf_bwqt42t80000gn/T/ipykernel_47900/436691358.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore_disk = Chroma(


In [44]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0.75,
    top_p=0.80,
    # You can omit google_api_key here because it will read from the env var,
    # but including it explicitly is fine too:
    google_api_key=os.environ["GOOGLE_API_KEY"]
)
print("LLM ready ✅")


LLM ready ✅


In [52]:
# 1. Gemini's embedding model, embedding-001
gemini_embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

# 2. Chroma Vector Database
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=gemini_embeddings,
    persist_directory="./chroma_db"
)

vectorstore_disk = Chroma(
    persist_directory="./chroma_db",
    embedding_function=gemini_embeddings
)
retriever = vectorstore_disk.as_retriever()

# 3. Gemini 2.0 Flash, gemini-2.0-flash
# temperature=0.75 and top_p=0.80 for model parameters
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0.75,
    top_p=0.80,
    google_api_key=os.environ["GOOGLE_API_KEY"]
)

In [53]:
# rag_chain - use invoke method to prompt grounded with the Presidential Proclomation documents
# RAG Prompt tempate
rag_prompt_template = """
You are a helpful assistant for answering questions. 
Use the following context to answer the question accurately.

Context:
{context}

Question:
{question}

Instructions:
- If the answer is not found in the context, respond with: "I could not find that information in the provided document."
- Keep your answer clear and under four sentences.

Answer:
"""
rag_prompt = PromptTemplate.from_template(rag_prompt_template)


In [54]:
# reference: https://python.langchain.com/v0.2/docs/tutorials/rag/
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [55]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

In [58]:
rag_chain.invoke("What are the learning objectives for Lesson 1?")

'In this lesson, we will learn what it means to have artificial intelligence. We will learn to tell if a device or item uses artificial intelligence.'

In [59]:
rag_chain.invoke("What are the recources for Lesson 1?")

'The resources for Lesson 1 are: Slide Deck, Robot Dog video, Self-Driving Car video, and optional Vocabulary cards.'